# Reproduce the random-2-per-FP pair dataset

This notebook rebuilds the (first party, third party) pair dataset that
feeds RQ2–RQ3 from scratch, using the same seed and selection rule as the
canonical pipeline. We then recompute every aggregate metric reported for
this pair set.

The construction rule is straightforward: for each qualified first party,
pick up to **two** of its embedded third parties uniformly at random
(seed = 42). First parties with fewer than two qualifying third parties
contribute what they have; first parties with zero are dropped.

The reference output lives in `data/random2_summary.json` (the canonical
summary) and `data/random2_pair_ids.json` (the full list of
`(first-party, third-party)` pair identifiers), so the last few cells can
check our reproduction line-by-line against the original.

Run top to bottom (Cell → Run All). Expect ~1 minute once the archive is
extracted.

> *Note on field names.* The upstream pair file uses JSON field names
> `site_etld1` and `vendor_etld1` — we keep those identifiers unchanged so
> the reproduction stays bit-identical. In the prose and printed output
> below we always say *first-party* and *third-party* for the same things.


## 1. Setup


### 1.1 Decompress the bundled dataset
Same tarball as the other notebook. Safe to re-run.


In [ ]:
import os, tarfile, pathlib

REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebook' else pathlib.Path.cwd()
DATA_DIR  = REPO_ROOT / 'data' / 'raw'
BUNDLE    = REPO_ROOT / 'data' / 'dataset.tar.gz'

DATA_DIR.mkdir(parents=True, exist_ok=True)
if (DATA_DIR / 'results.jsonl').exists():
    print(f'Already extracted at {DATA_DIR}')
else:
    assert BUNDLE.exists(), f'Missing {BUNDLE}.'
    print(f'Extracting {BUNDLE.name} ...')
    with tarfile.open(BUNDLE, 'r:gz') as tf:
        tf.extractall(DATA_DIR)
    print('Done.')


### 1.2 Imports and helpers


In [ ]:
import json, re, glob, random, collections
import numpy as np
import matplotlib.pyplot as plt

MIN_WORDS    = 500
K_PER_FP     = 2
SEED         = 42
WORD_RE      = re.compile(r'\S+')

def wc_en(entry):
    '''Matches the canonical pair builder exactly: always recompute from
    the cached text, never read a pre-stored word_count. Entries can have
    a cached word_count that disagrees with a fresh regex count by a word
    or two, which is enough to flip the >=500 threshold.'''
    if not isinstance(entry, dict):
        return None
    if entry.get('language', 'en') != 'en':
        return None
    return len(WORD_RE.findall(entry.get('text') or ''))

def pct(xs, p):
    xs = sorted(xs); k = (len(xs) - 1) * p / 100
    f = int(k); c = min(f + 1, len(xs) - 1)
    return xs[f] + (xs[c] - xs[f]) * (k - f)


### 1.3 Load the underlying dataset
(Same inputs as `reproduce_findings.ipynb`.)


In [ ]:
rows = [json.loads(ln) for ln in open(DATA_DIR / 'results.jsonl')]
print(f'rows in results.jsonl               : {len(rows):>7,}')

# Match the canonical pair builder's glob iteration order exactly.
# dict.update() overwrites earlier keys, so when a URL appears in multiple
# shards (e.g. a main shard plus an augmentation shard) the last one wins.
# Sorting would change which shard lands last and can flip the qualifying
# set by a few entries, which then cascades through random.sample.
tp_cache = {}
for f in glob.glob(str(DATA_DIR / 'results_shard*.tp_cache.json')):
    tp_cache.update(json.load(open(f)))
print(f'entries in third-party policy cache : {len(tp_cache):>7,}')

bl = json.load(open(DATA_DIR / 'policy_quality_blacklist.json'))
fp_blacklist      = set(bl.get('fp_blacklist_etld1', []))
tp_blacklist_urls = set(bl.get('tp_blacklist_urls', []))

redisc = {e['rediscovered_from_etld1'].lower(): u
          for u, e in tp_cache.items()
          if isinstance(e, dict) and e.get('rediscovered_from_etld1')}

qual_urls = {u for u, e in tp_cache.items()
             if (wc_en(e) or 0) >= MIN_WORDS and u not in tp_blacklist_urls}
print(f'qualifying TP policy URLs           : {len(qual_urls):>7,}')


### 1.4 Load the canonical reference output
`random2_pair_ids.json` lists the (first-party eTLD+1, third-party eTLD+1)
tuples produced by the canonical pipeline. `random2_summary.json` carries
the aggregate counters. We load both up-front so later cells can
cross-check against them.


In [ ]:
canonical_pairs   = json.load(open(REPO_ROOT / 'data' / 'random2_pair_ids.json'))
canonical_summary = json.load(open(REPO_ROOT / 'data' / 'random2_summary.json'))

CAN_FPS_EMITTED   = canonical_summary['sites_emitted']
CAN_PAIRS_EMITTED = canonical_summary['pairs_emitted']
CAN_ZERO_TP_FPS   = canonical_summary['sites_with_zero_valid_tp']
CAN_ONE_TP_FPS    = canonical_summary['sites_with_one_valid_tp']
CAN_QUAL_FPS      = canonical_summary['qualifying_fp_sites']
CAN_DIST_TP_ETLD  = canonical_summary['distinct_vendor_etld1']
CAN_DIST_TP_ENT   = canonical_summary['distinct_vendor_entities']

print(f'Canonical reference loaded')
print(f'  qualifying FPs           : {CAN_QUAL_FPS:,}')
print(f'  FPs emitted              : {CAN_FPS_EMITTED:,}')
print(f'  pairs emitted            : {CAN_PAIRS_EMITTED:,}')
print(f'  distinct TP eTLD+1s      : {CAN_DIST_TP_ETLD:,}')
print(f'  distinct TP entities     : {CAN_DIST_TP_ENT:,}')


## 2. Which first parties are eligible?

Only first parties that (a) qualify (English >=500w, not blacklisted) **and**
(b) have a non-empty inline first-party policy text in `results.jsonl` can
contribute pairs. The canonical pipeline also falls back to
`artifacts/<etld1>/policy.txt`, which is not shipped with this package —
but inline text is present for every qualifying first party so the two
paths agree here.


In [ ]:
qualifying_fp = {}   # fp_etld1 -> (text, wc)
for r in rows:
    if r.get('status') != 'ok' or not r.get('policy_is_english'):
        continue
    w = r.get('first_party_policy_word_count') or 0
    if w < MIN_WORDS:
        continue
    et = (r.get('site_etld1') or '').lower()
    if not et or et in fp_blacklist:
        continue
    text = r.get('first_party_policy') or ''
    if not text:
        continue
    qualifying_fp[et] = (text, w)

print(f'Qualifying FPs with inline text: {len(qualifying_fp):,}   [canonical {CAN_QUAL_FPS:,}]')


## 3. Pair construction funnel

For each qualifying first party we collect its qualifying third parties
(unique by eTLD+1, excluding its own eTLD+1), then sample up to
`K_PER_FP = 2` of them. This section is the funnel: how many third parties
each first party has available → how many first parties emit 0 / 1 / 2
pairs → total pairs emitted.


### 3.1 Distribution of qualifying TPs per first party


In [ ]:
per_fp_tp_lists = {}

for r in rows:
    if r.get('status') != 'ok' or not r.get('policy_is_english'):
        continue
    fp_et = (r.get('site_etld1') or '').lower()
    if fp_et not in qualifying_fp:
        continue
    tps = []
    seen = set()
    for tp in r.get('third_parties') or []:
        if not isinstance(tp, dict):
            continue
        tet = (tp.get('third_party_etld1') or '').lower()
        if not tet or tet in seen or tet == fp_et:
            continue
        pu = tp.get('policy_url') or redisc.get(tet)
        if not pu or pu in tp_blacklist_urls or pu not in qual_urls:
            continue
        entry = tp_cache.get(pu)
        if not isinstance(entry, dict):
            continue
        tp_wc = len(WORD_RE.findall(entry.get('text') or ''))
        if tp_wc < MIN_WORDS:
            continue
        seen.add(tet)
        tps.append((tet, pu, tp_wc, tp))
    per_fp_tp_lists[fp_et] = tps

counts = collections.Counter(len(v) for v in per_fp_tp_lists.values())
buckets = [0, 1, 2, 3, 5, 10, 20, 10**9]
print('Qualifying-TP-per-FP distribution (over the {:,} qualifying FPs)\n'.format(len(qualifying_fp)))
for lo, hi in zip(buckets[:-1], buckets[1:]):
    n = sum(c for k, c in counts.items() if lo <= k < hi)
    tag = f'{lo}' if hi == lo + 1 else (f'{lo}-{hi-1}' if hi < 10**9 else f'>={lo}')
    print(f'  {tag:<10s}  {n:>4}')

fps_zero = sum(1 for v in per_fp_tp_lists.values() if len(v) == 0)
fps_one  = sum(1 for v in per_fp_tp_lists.values() if len(v) == 1)
fps_gt1  = sum(1 for v in per_fp_tp_lists.values() if len(v) >= 2)
print()
print(f'FPs with 0 qualifying TPs (dropped): {fps_zero:>4}   [canonical {CAN_ZERO_TP_FPS}]')
print(f'FPs with exactly 1 qualifying TP   : {fps_one:>4}   [canonical {CAN_ONE_TP_FPS}]')
print(f'FPs with >=2 qualifying TPs        : {fps_gt1:>4}')
print(f'FPs that will emit at least 1 pair : {fps_one + fps_gt1:>4}   [canonical {CAN_FPS_EMITTED:,}]')
print(f'pairs expected (1*{fps_one} + 2*{fps_gt1}): '
      f'{fps_one + 2*fps_gt1:>5}   [canonical {CAN_PAIRS_EMITTED:,}]')


### 3.2 Deterministic sampling (seed = 42)

Two things make this reproducible:

- `random.seed(42)` is set once before the iteration starts.
- First parties are visited in `results.jsonl` order, which is the same
  order the canonical pipeline used.

For each eligible first party we call `random.sample(tps, k)` with
`k = min(2, len(tps))`. The call is skipped entirely for first parties
with zero qualifying TPs, matching the canonical logic.


In [ ]:
random.seed(SEED)
reproduced_pairs = []

for r in rows:
    if r.get('status') != 'ok' or not r.get('policy_is_english'):
        continue
    fp_et = (r.get('site_etld1') or '').lower()
    if fp_et not in qualifying_fp:
        continue
    tps = per_fp_tp_lists.get(fp_et, [])
    if not tps:
        continue
    k = min(K_PER_FP, len(tps))
    picks = random.sample(tps, k)
    for tet, pu, tp_wc, tp in picks:
        reproduced_pairs.append({
            'site_etld1': fp_et,
            'vendor_etld1': tet,
            'vendor_entity': tp.get('entity') or '',
            'vendor_policy_url': pu,
            'site_policy_word_count': qualifying_fp[fp_et][1],
            'vendor_policy_word_count': tp_wc,
        })

fps_in_pairs = len({p['site_etld1'] for p in reproduced_pairs})
print(f'Reproduced FPs emitted   : {fps_in_pairs:>5,}   [canonical {CAN_FPS_EMITTED:,}]')
print(f'Reproduced pairs emitted : {len(reproduced_pairs):>5,}   [canonical {CAN_PAIRS_EMITTED:,}]')


## 4. Verify against the canonical pair list

If our reproduction above is correct, the set of `(first-party, third-party)`
pairs should match the canonical one exactly.


In [ ]:
repro_set     = {(p['site_etld1'], p['vendor_etld1']) for p in reproduced_pairs}
canonical_set = {(p['site_etld1'], p['vendor_etld1']) for p in canonical_pairs}

print(f'Set sizes')
print(f'  canonical : {len(canonical_set):,}')
print(f'  reproduced: {len(repro_set):,}')
print()
print(f'Set comparison')
print(f'  common            : {len(canonical_set & repro_set):,}')
print(f'  only in canonical : {len(canonical_set - repro_set):,}')
print(f'  only in reproduced: {len(repro_set - canonical_set):,}')
if canonical_set == repro_set:
    print('\n✓ Exact match — reproduction is bit-identical to the canonical pair set.')
else:
    print('\n✗ Sets differ. Showing up to 5 pairs from each side:')
    for p in list(canonical_set - repro_set)[:5]:  print('  only canonical :', p)
    for p in list(repro_set - canonical_set)[:5]:  print('  only reproduced:', p)


## 5. Pair-level aggregate metrics

Reproduces the counters in `random2_summary.json`: most-sampled
third-party eTLD+1s and entities, and the top-10 concentration share.


### 5.1 Top-10 third-party eTLD+1s


In [ ]:
tp_counts = collections.Counter(p['vendor_etld1'] for p in reproduced_pairs)
print('Top 10 third-party eTLD+1s, by pair count')
print(f"{'rank':<5}{'repro':<10}{'canon':<10}{'third-party eTLD+1':<30}")
for i, (tp_ref, c_ref) in enumerate(canonical_summary['top_vendor_etld1'][:10], 1):
    c_rep = tp_counts.get(tp_ref, 0)
    ok = '✓' if c_rep == c_ref else '✗'
    print(f'{i:<5}{c_rep:<10,}{c_ref:<10,}{tp_ref:<30}{ok}')

print(f'\nDistinct third-party eTLD+1s   : {len(tp_counts):>4}   [canonical {CAN_DIST_TP_ETLD}]')


### 5.2 Top-10 third-party entities


In [ ]:
entity_counts = collections.Counter(p['vendor_entity'] for p in reproduced_pairs if p['vendor_entity'])
print('Top 10 third-party entities, by pair count')
print(f"{'rank':<5}{'repro':<10}{'canon':<10}{'entity':<30}")
for i, (e_ref, c_ref) in enumerate(canonical_summary['top_vendor_entities'][:10], 1):
    c_rep = entity_counts.get(e_ref, 0)
    ok = '✓' if c_rep == c_ref else '✗'
    print(f'{i:<5}{c_rep:<10,}{c_ref:<10,}{e_ref:<30}{ok}')

print(f'\nDistinct third-party entities  : {len(entity_counts):>4}   [canonical {CAN_DIST_TP_ENT}]')


### 5.3 Top-10 concentration share


In [ ]:
top10_etld1 = sum(c for _, c in tp_counts.most_common(10))
top10_ent   = sum(c for _, c in entity_counts.most_common(10))
n = len(reproduced_pairs)
print(f'Top-10 TP eTLD+1 share : {100*top10_etld1/n:5.1f}%   ({top10_etld1:,} / {n:,})')
print(f'Top-10 TP entity share : {100*top10_ent/n:5.1f}%   ({top10_ent:,} / {n:,})')


## 6. Policy-length distributions on the pair dataset

First-party policy length across the emitted first parties, and
third-party policy length across the emitted pairs (note: the same
third party can appear in many pairs, so the TP distribution is weighted
by how often each third party is sampled).


In [ ]:
fp_wcs_in_pairs = {p['site_etld1']: p['site_policy_word_count'] for p in reproduced_pairs}
tp_wcs_in_pairs = [p['vendor_policy_word_count'] for p in reproduced_pairs]

fp_wcs = list(fp_wcs_in_pairs.values())
print(f'First-party (distinct FPs in pairs)   n={len(fp_wcs):>5,}   '
      f'median={pct(fp_wcs,50):>6,.0f}   mean={sum(fp_wcs)/len(fp_wcs):>6,.0f}   '
      f'IQR={pct(fp_wcs,25):>5,.0f}-{pct(fp_wcs,75):<5,.0f}')
print(f'Third-party (one row per pair)        n={len(tp_wcs_in_pairs):>5,}   '
      f'median={pct(tp_wcs_in_pairs,50):>6,.0f}   mean={sum(tp_wcs_in_pairs)/len(tp_wcs_in_pairs):>6,.0f}   '
      f'IQR={pct(tp_wcs_in_pairs,25):>5,.0f}-{pct(tp_wcs_in_pairs,75):<5,.0f}')


## 7. Figure — top-10 third-party concentration

Horizontal bar chart of the ten most-sampled third-party eTLD+1s, using
the same compact publication style the other notebook uses.


In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'serif'],
    'axes.linewidth': 0.8,
    'axes.edgecolor': '#cccccc',
    'pdf.fonttype': 42,
})
PRIMARY = '#2171b5'

top = tp_counts.most_common(10)[::-1]
labels = [t for t, _ in top]
counts = [c for _, c in top]

fig, ax = plt.subplots(figsize=(3.35, 2.8))
y = np.arange(len(labels))
ax.barh(y, counts, color=PRIMARY, edgecolor='#0b3d6b', linewidth=0.5, height=0.68)
for yi, c in zip(y, counts):
    ax.text(c + max(counts)*0.01, yi, f'{c:,}', va='center', ha='left', fontsize=6.5)
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=7)
ax.set_xlabel('Number of pairs', fontsize=7.5)
ax.tick_params(axis='x', labelsize=7); ax.tick_params(axis='y', pad=2)
ax.grid(True, axis='x', alpha=0.25, linestyle=':', linewidth=0.5)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.set_xlim(0, max(counts) * 1.2)
plt.tight_layout(pad=0.3); plt.show()


## 8. Extraction manifest

The pair set above is the *selection* stage; the *extraction* stage runs
an LLM reader over every unique policy document that appears anywhere in
the pairs. Two pairs that reference the same third-party policy URL only
need that URL extracted once. Two different URLs that happen to serve
byte-identical text collapse further under SHA deduplication. This
section reports both counts and compares them to the canonical manifest
shipped in `data/manifest.csv`.

> *Why we read the canonical SHAs here.* First-party policy text is kept
> in `artifacts/<etld1>/policy.txt` files on HPC, which are too large to
> ship in this package. The pair set itself is enough to reproduce the
> *row-level* manifest (one row per distinct first party plus one per
> distinct third-party URL), which we do below. The *text-level*
> deduplication (SHA-16 over each policy text) cannot be recomputed from
> the bundled data alone — so we read the canonical SHA-16 column
> directly out of `manifest.csv` to report the dedup stats.


### 8.1 Row-level manifest (one row per unique document)

Derived purely from the reproduced pair set:
- **FP rows** — one per distinct `site_etld1`.
- **TP rows** — one per distinct `vendor_policy_url` (multiple TP
  eTLD+1s can collapse onto the same URL, e.g. `facebook.net` and
  `facebook.com` both pointing at Facebook's single privacy page).


In [ ]:
can_mf = json.load(open(REPO_ROOT / 'data' / 'manifest.summary.json'))

unique_fps_in_pairs = {p['site_etld1']      for p in reproduced_pairs}
unique_tp_urls      = {p['vendor_policy_url'] for p in reproduced_pairs}
manifest_rows       = len(unique_fps_in_pairs) + len(unique_tp_urls)

print(f'Unique first parties in the pair set      : {len(unique_fps_in_pairs):>5,}   [canonical {can_mf["unique_fp_sites"]}]')
print(f'Unique third-party policy URLs            : {len(unique_tp_urls):>5,}   [canonical {can_mf["unique_tp_policy_urls"]}]')
print(f'Manifest rows (FP rows + TP rows)         : {manifest_rows:>5,}   [canonical {can_mf["manifest_rows"]}]')


### 8.2 Text-level deduplication (read from `manifest.csv`)

Each row of `manifest.csv` carries a `sha256_16` column — a 16-hex-char
prefix of the text SHA. Counting distinct SHAs per role and across roles
gives the actual size of the extraction workload (the number of LLM
calls the extractor needs to make).


In [ ]:
import csv

fp_sha_rows = []
tp_sha_rows = []
with open(REPO_ROOT / 'data' / 'manifest.csv') as fh:
    for row in csv.DictReader(fh):
        if row['policy_source'] == 'first_party':
            fp_sha_rows.append(row)
        else:
            tp_sha_rows.append(row)

fp_shas = {r['sha256_16'] for r in fp_sha_rows}
tp_shas = {r['sha256_16'] for r in tp_sha_rows}
all_shas = fp_shas | tp_shas

print(f'Unique FP texts (SHA-dedup)              : {len(fp_shas):>5,}   [canonical {can_mf["unique_fp_texts"]}]')
print(f'Unique TP texts (SHA-dedup)              : {len(tp_shas):>5,}   [canonical {can_mf["unique_tp_texts"]}]')
print(f'Unique policies to extract (by role)     : {len(fp_shas) + len(tp_shas):>5,}   [canonical {can_mf["unique_policies_to_extract_by_role"]}]')
print(f'Unique policies collapsed across roles   : {len(all_shas):>5,}   [canonical {can_mf["unique_policies_collapsed_across_roles"]}]')

cross = fp_shas & tp_shas
print(f'\nTexts that appear as both FP and TP      : {len(cross):>5,}   (each saves one extraction)')


### 8.3 Extraction workload size

The `manifest.csv` also records `word_count` per row, so we can summarise
how long the documents the extractor will see are, broken out by role.


In [ ]:
fp_wcs = [int(r['word_count']) for r in fp_sha_rows]
tp_wcs = [int(r['word_count']) for r in tp_sha_rows]

print('Document word counts across the manifest')
print(f'  FP rows (n={len(fp_wcs):>5,})  '
      f'median={pct(fp_wcs,50):>6,.0f}  mean={sum(fp_wcs)/len(fp_wcs):>6,.0f}  '
      f'p90={pct(fp_wcs,90):>6,.0f}  max={max(fp_wcs):>6,}')
print(f'  TP rows (n={len(tp_wcs):>5,})  '
      f'median={pct(tp_wcs,50):>6,.0f}  mean={sum(tp_wcs)/len(tp_wcs):>6,.0f}  '
      f'p90={pct(tp_wcs,90):>6,.0f}  max={max(tp_wcs):>6,}')
print()
total_words = sum(fp_wcs) + sum(tp_wcs)
print(f'Total text going through the extractor: {total_words:,} words '
      f'across {len(fp_wcs)+len(tp_wcs):,} documents')


### 8.4 Cross-check against the canonical pair set

The manifest was generated from the canonical pair file. Since our
reproduction of the pair set is bit-identical (cell 4), the set of
manifest-row keys derived from our reproduction should exactly match
the canonical manifest's `etld1` column on the FP side and `policy_file`
on the TP side (one row per first-party eTLD+1; one row per TP URL's
text target).


In [ ]:
canonical_fp_etld1s = {r['etld1'] for r in fp_sha_rows}
canonical_tp_policy_files = {r['policy_file'] for r in tp_sha_rows}

print(f'FP side — canonical FP eTLD+1s        : {len(canonical_fp_etld1s):,}')
print(f'FP side — reproduced FP eTLD+1s       : {len(unique_fps_in_pairs):,}')
print(f'FP side — shared                      : {len(canonical_fp_etld1s & unique_fps_in_pairs):,}')
if canonical_fp_etld1s == unique_fps_in_pairs:
    print('           ✓ FP row set matches canonical exactly')
else:
    print('           ✗ FP row sets differ')
    extra_c = canonical_fp_etld1s - unique_fps_in_pairs
    extra_r = unique_fps_in_pairs - canonical_fp_etld1s
    if extra_c: print('             only canonical :', sorted(extra_c)[:5])
    if extra_r: print('             only reproduced:', sorted(extra_r)[:5])

print(f'\nTP side — canonical TP URL files      : {len(canonical_tp_policy_files):,}')
print(f'TP side — reproduced TP URLs          : {len(unique_tp_urls):,}')
